<a href="https://colab.research.google.com/github/MaxMariusJacobs/Smart-Blind-Cane/blob/main/Yolofinaltraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# SCHRITT 1: TOOLS & ABHÄNGIGKEITEN INSTALLIEREN
# ==============================================================================
!pip install -q ultralytics roboflow pycocotools

import os
import shutil
import glob
import json
import yaml
from roboflow import Roboflow
from ultralytics.utils.downloads import download

WORKSPACE_DIR = "/content/unified_dataset"
os.makedirs(f"{WORKSPACE_DIR}/images/train", exist_ok=True)
os.makedirs(f"{WORKSPACE_DIR}/images/val", exist_ok=True)
os.makedirs(f"{WORKSPACE_DIR}/labels/train", exist_ok=True)
os.makedirs(f"{WORKSPACE_DIR}/labels/val", exist_ok=True)

# ==============================================================================
# SCHRITT 2: DATASET A (GEHWEGE) LADEN & BEREINIGEN
# ==============================================================================
ROBOFLOW_API_KEY = "JlQ2tiTxqxfCxBOqB5ER"  # <-- Hier Key eintragen!

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("u22programcontest").project("sidewalk-segmentation-ij8uy")
ds_sidewalk = project.version(1).download("yolov11")

# Bereinigungs-Mapping für Gehweg-Daten:
# Alt: 0: Roadway, 1: Sidewalk, 2: downstairs, 3: path, 4: sidewalk, 5: upstairs
# Neu: 0: sidewalk, 1: roadway, 2: path, 3: stairs
sidewalk_map = {1: 0, 4: 0, 0: 1, 3: 2, 2: 3, 5: 3}

def process_sidewalk_split(src_split, dst_split):
    img_files = glob.glob(f"{ds_sidewalk.location}/{src_split}/images/*.*")
    for img_path in img_files:
        base = os.path.splitext(os.path.basename(img_path))[0]
        ext = os.path.splitext(img_path)[1]
        lbl_path = f"{ds_sidewalk.location}/{src_split}/labels/{base}.txt"

        if not os.path.exists(lbl_path):
            continue

        with open(lbl_path, "r") as f:
            lines = f.readlines()

        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            old_cls = int(parts[0])
            if old_cls in sidewalk_map:
                parts[0] = str(sidewalk_map[old_cls])
                new_lines.append(" ".join(parts) + "\n")

        if new_lines:
            shutil.copyfile(img_path, f"{WORKSPACE_DIR}/images/{dst_split}/sw_{base}{ext}")
            with open(f"{WORKSPACE_DIR}/labels/{dst_split}/sw_{base}.txt", "w") as f:
                f.writelines(new_lines)

process_sidewalk_split("train", "train")
process_sidewalk_split("valid", "val")
print("[INFO] Gehweg-Daten bereinigt und kopiert.")

# ==============================================================================
# SCHRITT 3: DATASET B (COCO VAL2017) HERUNTERLADEN & FILTERN
# ==============================================================================
# Lädt val2017 (1GB) + Annotationen
download("http://images.cocodataset.org/zips/val2017.zip", dir="/content/coco", unzip=True)
download("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", dir="/content/coco", unzip=True)

from pycocotools.coco import COCO
coco = COCO("/content/coco/annotations/instances_val2017.json")

# COCO-IDs: 1: person -> 4, 2: bicycle -> 5, 3: car / 56: chair / 58: potted plant -> 6 (obstacle)
target_coco_categories = {
    1: 4,   # person
    2: 5,   # bicycle
    3: 6,   # car -> obstacle
    62: 6,  # chair -> obstacle
    64: 6   # potted plant -> obstacle
}

selected_img_ids = set()
for cat_id in target_coco_categories.keys():
    selected_img_ids.update(coco.getImgIds(catIds=[cat_id]))

# Beschränke auf 800 repräsentative Bilder um Disbalance zu vermeiden
selected_img_ids = list(selected_img_ids)[:800]
split_idx = int(len(selected_img_ids) * 0.8)
train_ids = selected_img_ids[:split_idx]
val_ids = selected_img_ids[split_idx:]

def process_coco_split(img_ids, split_name):
    for img_id in img_ids:
        img_info = coco.loadImgs(img_id)[0]
        w, h = img_info['width'], img_info['height']
        ann_ids = coco.getAnnIds(imgIds=img_id, iscrowd=False)
        anns = coco.loadAnns(ann_ids)

        yolo_lines = []
        for ann in anns:
            cat_id = ann['category_id']
            if cat_id not in target_coco_categories:
                continue
            if 'segmentation' not in ann or not isinstance(ann['segmentation'], list) or len(ann['segmentation']) == 0:
                continue

            new_cls = target_coco_categories[cat_id]
            # Normalisiere Polygone für YOLO Segmentation: x1 y1 x2 y2 ...
            poly = ann['segmentation'][0]
            if len(poly) < 6:
                continue
            norm_poly = []
            for i in range(0, len(poly), 2):
                norm_poly.append(f"{min(max(poly[i] / w, 0.0), 1.0):.6f}")
                norm_poly.append(f"{min(max(poly[i+1] / h, 0.0), 1.0):.6f}")
            yolo_lines.append(f"{new_cls} " + " ".join(norm_poly) + "\n")

        if yolo_lines:
            src_file = f"/content/coco/val2017/{img_info['file_name']}"
            dst_file = f"{WORKSPACE_DIR}/images/{split_name}/coco_{img_info['file_name']}"
            base = os.path.splitext(img_info['file_name'])[0]
            lbl_file = f"{WORKSPACE_DIR}/labels/{split_name}/coco_{base}.txt"

            shutil.copyfile(src_file, dst_file)
            with open(lbl_file, "w") as f:
                f.writelines(yolo_lines)

process_coco_split(train_ids, "train")
process_coco_split(val_ids, "val")
print("[INFO] 800 COCO-Bilder extrahiert, normalisiert und gemergt.")

# ==============================================================================
# SCHRITT 4: DATA.YAML & STATISTIK GENERIEREN
# ==============================================================================
yaml_data = {
    "path": WORKSPACE_DIR,
    "train": "images/train",
    "val": "images/val",
    "nc": 7,
    "names": ["sidewalk", "roadway", "path", "stairs", "person", "bicycle", "obstacle"]
}

with open(f"{WORKSPACE_DIR}/data.yaml", "w") as f:
    yaml.dump(yaml_data, f, default_flow_style=False)

# Validierungs-Zählung
class_counts = {name: 0 for name in yaml_data["names"]}
for lbl_path in glob.glob(f"{WORKSPACE_DIR}/labels/train/*.txt"):
    with open(lbl_path, "r") as f:
        for line in f:
            p = line.strip().split()
            if p:
                cls_idx = int(p[0])
                if cls_idx < len(yaml_data["names"]):
                    class_counts[yaml_data["names"][cls_idx]] += 1

print("\n--- DATENSATZ-ÜBERSICHT (TRAIN-SPLIT) ---")
for k, v in class_counts.items():
    print(f"  {k.ljust(10)}: {v} Instanzen")
print(f"Gespeicherter Datensatz-Pfad: {WORKSPACE_DIR}/data.yaml")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 5.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Sidewalk-Segmentation-1 in yolov11:: 100%|██████████| 3861/3861 [00:00<00:00, 7655.21it/s]


[INFO] Gehweg-Daten bereinigt und kopiert.
Unzipping /content/coco/val2017.zip to /content/coco/val2017...: 100% ━━━━━━━━━━━━ 5001/5001 1.1Kfiles/s 4.6s
Unzipping /content/coco/annotations_trainval2017.zip to /content/coco/annotations...: 100% ━━━━━━━━━━━━ 6/6 1.3s/files 7.6s
loading annotations into memory...
Done (t=0.49s)
creating index...
index created!
[INFO] 800 COCO-Bilder extrahiert, normalisiert und gemergt.

--- DATENSATZ-ÜBERSICHT (TRAIN-SPLIT) ---
  sidewalk  : 1114 Instanzen
  roadway   : 955 Instanzen
  path      : 552 Instanzen
  stairs    : 430 Instanzen
  person    : 2182 Instanzen
  bicycle   : 71 Instanzen
  obstacle  : 862 Instanzen
Gespeicherter Datensatz-Pfad: /content/unified_dataset/data.yaml


In [ ]:
# ==============================================================================
# 1. SETUP & ABHÄNGIGKEITEN
# ==============================================================================
!pip install -q ultralytics roboflow pycocotools

import os
import shutil
import glob
import yaml
import numpy as np
from roboflow import Roboflow
from ultralytics import YOLO
from ultralytics.utils.downloads import download
from pycocotools.coco import COCO

FINAL_DATASET = "/content/cross_labeled_dataset"
for split in ["train", "val"]:
    os.makedirs(f"{FINAL_DATASET}/images/{split}", exist_ok=True)
    os.makedirs(f"{FINAL_DATASET}/labels/{split}", exist_ok=True)

# ==============================================================================
# 2. MODELLE & DATEN LADEN
# ==============================================================================
# Gehweg-Experte: Nutze den besten 4-Klassen-Checkpoint aus train-2
sidewalk_checkpoint = glob.glob("/content/**/train-2/**/best.pt", recursive=True)
if not sidewalk_checkpoint:
    sidewalk_checkpoint = glob.glob("/content/**/runs/segment/**/best.pt", recursive=True)
sidewalk_model_path = sidewalk_checkpoint[0]
print(f"[INFO] Gehweg-Experte geladen von: {sidewalk_model_path}")
sidewalk_expert = YOLO(sidewalk_model_path)

# COCO-Experte: Großes Modell für hochpräzise Hindernis-Masken
coco_expert = YOLO("yolo11m-seg.pt")

# Gehweg-Datensatz von Roboflow holen
ROBOFLOW_API_KEY = "JlQ2tiTxqxfCxBOqB5ER"  # <-- Ersetzen!
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("u22programcontest").project("sidewalk-segmentation-ij8uy")
ds_sidewalk = project.version(1).download("yolov11")

# COCO val2017 für Hindernis-Diversität laden
download("http://images.cocodataset.org/zips/val2017.zip", dir="/content/coco", unzip=True)
download("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", dir="/content/coco", unzip=True)
coco = COCO("/content/coco/annotations/instances_val2017.json")

# ==============================================================================
# 3. ZIEL-KLASSEN-SCHEMA (7 KLASSEN)
# 0: sidewalk | 1: roadway | 2: path | 3: stairs | 4: person | 5: bicycle | 6: obstacle
# ==============================================================================
sidewalk_base_map = {1: 0, 4: 0, 0: 1, 3: 2, 2: 3, 5: 3}
coco_target_classes = {
    0: 4,   # person -> 4
    1: 5,   # bicycle -> 5
    2: 6,   # car -> 6
    3: 6,   # motorcycle -> 6
    5: 6,   # bus -> 6
    7: 6,   # truck -> 6
    56: 6,  # chair -> 6
    58: 6   # potted plant -> 6
}

# ==============================================================================
# 4. TEIL A: GEHWEG-BILDER LADEN + HINDERNISSE AUTO-LABELN
# ==============================================================================
print("\n[INFO] Phase 1: Auto-Labeling von Hindernissen auf Gehweg-Bildern...")

def enrich_sidewalk_images(src_split, dst_split):
    img_files = glob.glob(f"{ds_sidewalk.location}/{src_split}/images/*.*")
    for img_path in img_files:
        base = os.path.splitext(os.path.basename(img_path))[0]
        ext = os.path.splitext(img_path)[1]
        orig_lbl = f"{ds_sidewalk.location}/{src_split}/labels/{base}.txt"

        lines_to_keep = []
        if os.path.exists(orig_lbl):
            with open(orig_lbl, "r") as f:
                for line in f:
                    p = line.strip().split()
                    if not p: continue
                    cls_id = int(p[0])
                    if cls_id in sidewalk_base_map:
                        p[0] = str(sidewalk_base_map[cls_id])
                        lines_to_keep.append(" ".join(p) + "\n")

        # Inferenz mit COCO-Experte
        pred = coco_expert(img_path, conf=0.35, imgsz=640, verbose=False)[0]
        if pred.masks is not None:
            classes = pred.boxes.cls.cpu().numpy().astype(int)
            polys = pred.masks.xyn
            for cls_idx, poly in zip(classes, polys):
                if cls_idx in coco_target_classes and len(poly) >= 6:
                    target_cls = coco_target_classes[cls_idx]
                    flat_poly = " ".join([f"{coord:.6f}" for point in poly for coord in point])
                    lines_to_keep.append(f"{target_cls} {flat_poly}\n")

        # Speichern im Zielordner
        dst_img = f"{FINAL_DATASET}/images/{dst_split}/sw_{base}{ext}"
        dst_lbl = f"{FINAL_DATASET}/labels/{dst_split}/sw_{base}.txt"
        shutil.copyfile(img_path, dst_img)
        with open(dst_lbl, "w") as f:
            f.writelines(lines_to_keep)

enrich_sidewalk_images("train", "train")
enrich_sidewalk_images("valid", "val")
print("[INFO] Gehweg-Bilder erfolgreich mit Hindernissen annotiert.")

# ==============================================================================
# 5. TEIL B: COCO-BILDER FILTERN + GEHWEGE AUTO-LABELN
# ==============================================================================
print("\n[INFO] Phase 2: Auto-Labeling von Gehwegen auf COCO-Bildern...")

target_coco_ids = [1, 2, 3] # person, bicycle, car
coco_img_ids = set()
for c in target_coco_ids:
    coco_img_ids.update(coco.getImgIds(catIds=[c]))

# 400 scharfe Bilder für ausgewogenes Balancing
selected_coco_ids = list(coco_img_ids)[:400]
split_at = int(len(selected_coco_ids) * 0.8)
coco_splits = {"train": selected_coco_ids[:split_at], "val": selected_coco_ids[split_at:]}

for split_name, img_ids in coco_splits.items():
    for img_id in img_ids:
        info = coco.loadImgs(img_id)[0]
        w, h = info["width"], info["height"]
        src_img = f"/content/coco/val2017/{info['file_name']}"
        base = os.path.splitext(info["file_name"])[0]

        coco_lines = []
        ann_ids = coco.getAnnIds(imgIds=img_id, iscrowd=False)
        anns = coco.loadAnns(ann_ids)
        for ann in anns:
            cat = ann["category_id"]
            # COCO Categories -> Index-Mapping
            mapped_cat = None
            if cat == 1: mapped_cat = 4       # person
            elif cat == 2: mapped_cat = 5     # bicycle
            elif cat in [3, 4, 6, 8]: mapped_cat = 6 # vehicle -> obstacle

            if mapped_cat is not None and "segmentation" in ann and isinstance(ann["segmentation"], list) and len(ann["segmentation"]) > 0:
                poly = ann["segmentation"][0]
                if len(poly) >= 6:
                    norm = [f"{poly[i]/w:.6f} {poly[i+1]/h:.6f}" for i in range(0, len(poly), 2)]
                    coco_lines.append(f"{mapped_cat} " + " ".join(norm) + "\n")

        # Inferenz mit Gehweg-Experte
        sw_pred = sidewalk_expert(src_img, conf=0.40, imgsz=320, verbose=False)[0]
        if sw_pred.masks is not None:
            sw_classes = sw_pred.boxes.cls.cpu().numpy().astype(int)
            sw_polys = sw_pred.masks.xyn
            for c_id, poly in zip(sw_classes, sw_polys):
                if c_id in [0, 1, 2, 3] and len(poly) >= 6:
                    flat_poly = " ".join([f"{coord:.6f}" for point in poly for coord in point])
                    coco_lines.append(f"{c_id} {flat_poly}\n")

        if coco_lines:
            dst_img = f"{FINAL_DATASET}/images/{split_name}/coco_{info['file_name']}"
            dst_lbl = f"{FINAL_DATASET}/labels/{split_name}/coco_{base}.txt"
            shutil.copyfile(src_img, dst_img)
            with open(dst_lbl, "w") as f:
                f.writelines(coco_lines)

print("[INFO] COCO-Bilder erfolgreich mit Gehwegen annotiert.")

# ==============================================================================
# 6. DATA.YAML GENERIEREN & STATISTIK PRÜFEN
# ==============================================================================
data_yaml = {
    "path": FINAL_DATASET,
    "train": "images/train",
    "val": "images/val",
    "nc": 7,
    "names": ["sidewalk", "roadway", "path", "stairs", "person", "bicycle", "obstacle"]
}

with open(f"{FINAL_DATASET}/data.yaml", "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

stats = {name: 0 for name in data_yaml["names"]}
for lbl in glob.glob(f"{FINAL_DATASET}/labels/train/*.txt"):
    with open(lbl, "r") as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                idx = int(parts[0])
                if idx < len(data_yaml["names"]):
                    stats[data_yaml["names"][idx]] += 1

print("\n--- BEREINIGTE TRAININGS-STATISTIK ---")
for k, v in stats.items():
    print(f"  {k.ljust(10)}: {v} Instanzen")
print(f"\nBereit zum Training mit: {FINAL_DATASET}/data.yaml")

[INFO] Gehweg-Experte geladen von: /content/runs/segment/train/weights/best.pt
loading Roboflow workspace...
loading Roboflow project...
WARNING ⚠️ Skipping /content/coco/val2017.zip unzip as destination directory /content/coco/val2017 is not empty.
WARNING ⚠️ Skipping /content/coco/annotations_trainval2017.zip unzip as destination directory /content/coco/annotations is not empty.
loading annotations into memory...
Done (t=0.78s)
creating index...
index created!

[INFO] Phase 1: Auto-Labeling von Hindernissen auf Gehweg-Bildern...
[INFO] Gehweg-Bilder erfolgreich mit Hindernissen annotiert.

[INFO] Phase 2: Auto-Labeling von Gehwegen auf COCO-Bildern...
[INFO] COCO-Bilder erfolgreich mit Gehwegen annotiert.

--- BEREINIGTE TRAININGS-STATISTIK ---
  sidewalk  : 1115 Instanzen
  roadway   : 955 Instanzen
  path      : 552 Instanzen
  stairs    : 430 Instanzen
  person    : 1299 Instanzen
  bicycle   : 58 Instanzen
  obstacle  : 1864 Instanzen

Bereit zum Training mit: /content/cross_labe

In [ ]:
from ultralytics import YOLO

# 1. Vortrainiertes Nano-Segmentierungs-Backbone laden
model = YOLO("yolo11n-seg.pt")

# 2. Training auf dem cross-gelabelten 7-Klassen-Datensatz
results = model.train(
    data="/content/cross_labeled_dataset/data.yaml",
    epochs=50,
    imgsz=320,              # Zielauflösung für Pixel 7a GPU-Pipeline
    batch=16,               # Verhindert GPU-OOM bei Masken-Berechnung
    device=0,               # Tesla T4 GPU
    patience=12,            # Early Stopping bei Stagnation
    save=True,
    plots=True,
    workers=4,
    optimizer="auto",

    # --- Gezielte Augmentierung für Ego-Perspektive & Stock-Dynamik ---
    degrees=8.0,            # Simuliert Pendel- und Kippbewegungen des Stocks
    perspective=0.0004,     # Gleicht den ~35° Neigungswinkel nach unten an
    scale=0.5,              # Trainiert auf kleine / weit entfernte Objekte bei 320px
    mosaic=1.0,             # 4-Bild-Synthese für dichte Multi-Objekt-Szenen
    mixup=0.1,              # Bildüberblendung gegen Overfitting bei Pseudo-Labels
    close_mosaic=10         # Schaltet Mosaic ab Epoche 40 ab für messerscharfe Gehwegkanten
)

print("[INFO] Training abgeschlossen! Validierungs-Plots liegen unter runs/segment/train/")

Ultralytics 8.4.152 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/cross_labeled_dataset/data.yaml, degrees=8.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=None, opse

In [ ]:
import os
import shutil
import glob
import yaml
from pycocotools.coco import COCO

FINAL_DATASET = "/content/cross_labeled_dataset"
os.makedirs(f"{FINAL_DATASET}/images/train", exist_ok=True)
os.makedirs(f"{FINAL_DATASET}/images/val", exist_ok=True)
os.makedirs(f"{FINAL_DATASET}/labels/train", exist_ok=True)
os.makedirs(f"{FINAL_DATASET}/labels/val", exist_ok=True)

coco = COCO("/content/coco/annotations/instances_val2017.json")

# Gezielt Bilder mit Fahrrädern (ID 2) und Personen (ID 1) laden
bike_img_ids = coco.getImgIds(catIds=[2])
person_img_ids = coco.getImgIds(catIds=[1])

# 300 reine Fahrrad-Szenen + 200 Personen-Szenen priorisieren
selected_ids = list(set(bike_img_ids[:300] + person_img_ids[:200]))
split_at = int(len(selected_ids) * 0.8)
splits = {"train": selected_ids[:split_at], "val": selected_ids[split_at:]}

for split_name, ids in splits.items():
    for img_id in ids:
        info = coco.loadImgs(img_id)[0]
        w, h = info["width"], info["height"]
        src_img = f"/content/coco/val2017/{info['file_name']}"
        base = os.path.splitext(info["file_name"])[0]

        lines = []
        anns = coco.loadAnns(coco.getAnnIds(imgIds=img_id, iscrowd=False))
        for ann in anns:
            cat = ann["category_id"]
            mapped = None
            if cat == 1: mapped = 4       # person
            elif cat == 2: mapped = 5     # bicycle
            elif cat in [3, 4, 6, 8]: mapped = 6 # vehicle -> obstacle

            if mapped is not None and "segmentation" in ann and isinstance(ann["segmentation"], list) and len(ann["segmentation"]) > 0:
                poly = ann["segmentation"][0]
                if len(poly) >= 6:
                    norm = [f"{poly[i]/w:.6f} {poly[i+1]/h:.6f}" for i in range(0, len(poly), 2)]
                    lines.append(f"{mapped} " + " ".join(norm) + "\n")

        # Auto-Label Gehwege auf diesen COCO-Bildern
        sw_pred = sidewalk_expert(src_img, conf=0.35, imgsz=320, verbose=False)[0]
        if sw_pred.masks is not None:
            classes = sw_pred.boxes.cls.cpu().numpy().astype(int)
            polys = sw_pred.masks.xyn
            for c_id, poly in zip(classes, polys):
                if c_id in [0, 1, 2, 3] and len(poly) >= 6:
                    flat_poly = " ".join([f"{coord:.6f}" for point in poly for coord in point])
                    lines.append(f"{c_id} {flat_poly}\n")

        if lines:
            shutil.copyfile(src_img, f"{FINAL_DATASET}/images/{split_name}/boost_{info['file_name']}")
            with open(f"{FINAL_DATASET}/labels/{split_name}/boost_{base}.txt", "w") as f:
                f.writelines(lines)

print("[INFO] Fahrrad- und Personen-Boost erfolgreich in Datensatz integriert!")

loading annotations into memory...
Done (t=0.41s)
creating index...
index created!
[INFO] Fahrrad- und Personen-Boost erfolgreich in Datensatz integriert!


In [ ]:
from ultralytics import YOLO

# 1. Vortrainiertes Standard-Modell laden (enthält COCO-Wissen)
model = YOLO("yolo11n-seg.pt")

# 2. Feintuning mit geschütztem Backbone
results = model.train(
    data="/content/cross_labeled_dataset/data.yaml",
    epochs=40,
    imgsz=320,
    batch=16,
    device=0,
    patience=10,
    save=True,
    plots=True,

    # --- Schutz des vortrainierten Wissens ---
    freeze=10,             # Friert die ersten 10 Schichten (Backbone) ein!
    lr0=0.003,             # Sanftere Lernrate (verhindert Überschreiben)
    lrf=0.01,              # Finaler Lernraten-Faktor

    # --- Gezielte Verlust-Gewichtung ---
    cls=1.8,               # Erhöht Klassifikations-Priorität drastisch (Standard: 0.5)
    box=8.5,               # Schärft Box-Positionierung für schmale Objekte (Standard: 7.5)

    # --- Geometrische Anpassung ---
    scale=0.4,             # Verhindert Überanpassung an Riesenskalen
    perspective=0.0003,
    mosaic=1.0,
    close_mosaic=8
)

print("[INFO] Feintuning abgeschlossen!")

Ultralytics 8.4.152 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=8.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=8, cls=1.8, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/cross_labeled_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.003, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-3, nbs=64, nms=None, opset=

In [ ]:
!pip install -q ultralytics roboflow

import os
import shutil
import glob
import yaml
from roboflow import Roboflow

SURFACE_DIR = "/content/surface_dataset"
for split in ["train", "val"]:
    os.makedirs(f"{SURFACE_DIR}/images/{split}", exist_ok=True)
    os.makedirs(f"{SURFACE_DIR}/labels/{split}", exist_ok=True)

# 1. Gehweg-Datensatz laden
ROBOFLOW_API_KEY = "JlQ2tiTxqxfCxBOqB5ER"  # <-- Hier Key eintragen!
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("u22programcontest").project("sidewalk-segmentation-ij8uy")
ds = project.version(1).download("yolov11")

# 2. Bereinigtes Mapping:
# 0: Roadway -> 1, 1: Sidewalk -> 0, 2: downstairs -> 3, 3: path -> 2, 4: sidewalk -> 0, 5: upstairs -> 3
label_map = {1: 0, 4: 0, 0: 1, 3: 2, 2: 3, 5: 3}

def prepare_split(src_split, dst_split):
    img_files = glob.glob(f"{ds.location}/{src_split}/images/*.*")
    for img_path in img_files:
        base = os.path.splitext(os.path.basename(img_path))[0]
        ext = os.path.splitext(img_path)[1]
        lbl_path = f"{ds.location}/{src_split}/labels/{base}.txt"

        if not os.path.exists(lbl_path):
            continue

        with open(lbl_path, "r") as f:
            lines = f.readlines()

        remapped_lines = []
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            old_cls = int(parts[0])
            if old_cls in label_map:
                parts[0] = str(label_map[old_cls])
                remapped_lines.append(" ".join(parts) + "\n")

        if remapped_lines:
            shutil.copyfile(img_path, f"{SURFACE_DIR}/images/{dst_split}/{base}{ext}")
            with open(f"{SURFACE_DIR}/labels/{dst_split}/{base}.txt", "w") as f:
                f.writelines(remapped_lines)

prepare_split("train", "train")
prepare_split("valid", "val")

# 3. data.yaml schreiben
yaml_content = {
    "path": SURFACE_DIR,
    "train": "images/train",
    "val": "images/val",
    "nc": 4,
    "names": ["sidewalk", "roadway", "path", "stairs"]
}

with open(f"{SURFACE_DIR}/data.yaml", "w") as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print("[INFO] Datensatz erfolgreich auf 4 Untergrund-Klassen bereinigt!")

loading Roboflow workspace...
loading Roboflow project...
[INFO] Datensatz erfolgreich auf 4 Untergrund-Klassen bereinigt!


In [ ]:
from ultralytics import YOLO

# 1. Vortrainiertes Segmentierungsmodell laden
model = YOLO("yolo11n-seg.pt")

# 2. Gezieltes Training für Bodengeometrie
results = model.train(
    data="/content/surface_dataset/data.yaml",
    epochs=60,
    imgsz=320,
    batch=16,
    device=0,
    patience=15,
    save=True,
    plots=True,
    optimizer="auto",
    cos_lr=True,            # Glatteres Ausklingen der Lernrate

    # --- Verlust-Gewichtung ---
    cls=1.2,                # Verstärkt die Strafe für verpasste Treppen
    box=7.5,

    # --- Geometrie-Augmentation für Gehwege ---
    perspective=0.0006,     # Simuliert den Blickwinkel von oben (~35°)
    degrees=6.0,            # Leichte Pendelbewegung
    scale=0.4,              # Trainiert Nah- und Fernbereich
    fliplr=0.5,             # Horizontales Spiegeln (links/rechts irrelevant)
    flipud=0.0,             # KEIN vertikales Spiegeln! Schwerkraft für Treppen erhalten
    mosaic=1.0,
    close_mosaic=10         # Exakte Maskenkanten in den letzten 10 Epochen
)

print("[INFO] Training abgeschlossen!")

Ultralytics 8.4.152 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=1.2, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/surface_dataset/data.yaml, degrees=6.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-4, nbs=64, nms=None, opset=None,

In [ ]:
import glob
from google.colab import files

# Neuesten Checkpoint laden
checkpoint = glob.glob("runs/segment/**/weights/best.pt", recursive=True)[-1]
trained_model = YOLO(checkpoint)

# Exportieren für GPU-Shader-Kerne (FP32 TFLite)
export_file = trained_model.export(
    format="tflite",
    imgsz=320,
    simplify=True,
    nms=False
)

# Automatisch herunterladen
tflite_files = glob.glob("runs/segment/**/weights/**/*.tflite", recursive=True) + glob.glob("runs/segment/**/weights/*.tflite", recursive=True)
files.download(max(tflite_files, key=os.path.getmtime))

WARNING ⚠️ format='tflite' is deprecated as of 8.4.83 and has been replaced by the unified Google LiteRT format. Exporting format='litert' instead. See https://docs.ultralytics.com/integrations/litert
Ultralytics 8.4.152 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ This model has no one-to-one head; using one-to-many outputs.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO11n-seg summary (fused): 113 layers, 2,835,348 parameters, 0 gradients, 2.4 GFLOPs

PyTorch: starting from 'runs/segment/train-4/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) ((1, 40, 2100), (1, 32, 80, 80)) (5.7 MB)
requirements: Ultralytics requirements ['litert-torch>=0.9.0', 'ai-edge-litert>=2.1.4'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 86 packages in 811ms
Prepared 15 packages in 3.28s
Uninstalled 3 package

invalid escape sequence '\.'



LiteRT: starting export with litert_torch 0.9.4...


(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

...

(00:00) [START] Write Model to runs/segment/train-4/weights/best.tflite

(00:00) [ DONE] Write Model to runs/segment/train-4/weights/best.tflite (+00:00)

LiteRT: export success ✅ 28.5s, saved as 'runs/segment/train-4/weights/best.tflite' (11.1 MB)

Export complete (29.2s)
Results saved to /content/runs/segment/train-4/weights/best.tflite
Predict:         yolo predict task=segment model=runs/segment/train-4/weights/best.tflite imgsz=320 
Validate:        yolo val task=segment model=runs/segment/train-4/weights/best.tflite imgsz=320 data=/content/surface_dataset/data.yaml  
Visualize:       https://netron.app


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install -q ultralytics onnx onnxslim

import os
import glob
import torch
import torch.nn as nn
from ultralytics import YOLO
from google.colab import files

# 1. Vortrainiertes Standard-YOLO11n laden (Detection, kein Segmentierungs-Overhead)
model = YOLO("yolo11n.pt")
orig_names = model.names

# 2. Relevante Zielklassen definieren (COCO-IDs)
# 0: person, 1: bicycle, 2: car, 3: motorcycle, 5: bus, 7: truck, 13: bench, 16: dog
TARGET_INDICES = [0, 1, 2, 3, 5, 7, 13, 16]
new_names = {i: orig_names[idx] for i, idx in enumerate(TARGET_INDICES)}
new_nc = len(TARGET_INDICES)

print(f"[INFO] Ursprüngliche Klassen: {len(orig_names)} -> Beschnitten auf: {new_nc}")
for i, name in new_names.items():
    print(f"  Index {i}: {name}")

# 3. Graph-Surgery am Detect-Kopf durchführen
detect_head = model.model.model[-1]
old_nc = detect_head.nc

# Beschnitt der 1x1 Convolutions in den 3 Skalierungsstufen (Stride 8, 16, 32)
for branch in detect_head.cv3:
    for m in branch.modules():
        if isinstance(m, nn.Conv2d) and m.out_channels == old_nc:
            # Gewichte und Bias exakt auf die 8 Ziel-Indizes slicen
            m.weight = nn.Parameter(m.weight.data[TARGET_INDICES].clone())
            if m.bias is not None:
                m.bias = nn.Parameter(m.bias.data[TARGET_INDICES].clone())
            m.out_channels = new_nc

# Metadaten des Modells aktualisieren
detect_head.nc = new_nc
detect_head.no = new_nc + detect_head.reg_max * 4
model.model.nc = new_nc
model.model.names = new_names
model.names = new_names

# 4. Bereinigtes Modell speichern
pruned_pt_path = "/content/yolo11n_obstacles_pruned.pt"
torch.save(model.model.state_dict(), "/content/pruned_weights.pt")
model.save(pruned_pt_path)
print(f"[INFO] Beschnittenes PyTorch-Modell gespeichert: {pruned_pt_path}")

# 5. Export als schlankes TFLite FlatBuffer (320x320 FP32 für GPU-Delegate)
pruned_model = YOLO(pruned_pt_path)
export_path = pruned_model.export(
    format="tflite",
    imgsz=320,
    simplify=True,
    nms=False
)

# 6. Herunterladen
tflite_candidates = glob.glob("/content/**/yolo11n_obstacles_pruned*.tflite", recursive=True) + \
                    glob.glob("/content/**/weights/*.tflite", recursive=True)

if tflite_candidates:
    target_file = max(tflite_candidates, key=os.path.getmtime)
    print(f"[SUCCESS] Lade exportiertes Modell herunter: {target_file}")
    files.download(target_file)
else:
    print(f"[HINWEIS] Manuell herunterladen von: {export_path}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.0/239.0 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xdsl 0.28.0 requires typing-extensions<4.13,>=4.7, but you have typing-extensions 4.16.0 which is incompatible.
[INFO] Ursprüngliche Klassen: 80 -> Beschnitten auf: 8
  Index 0: person
  Index 1: bicycle
  Index 2: car
  Index 3: motorcycle
  Index 4: bus
  Index 5: truck
  Index 6: bench
  Index 7: dog


AttributeError: property 'names' of 'YOLO' object has no setter

In [ ]:
import os
import glob
import torch
import torch.nn as nn
from ultralytics import YOLO
from google.colab import files

# 1. Vortrainiertes Standard-YOLO11n laden
model = YOLO("yolo11n.pt")
orig_names = model.names

# 2. Relevante Zielklassen definieren (COCO-IDs)
# 0: person, 1: bicycle, 2: car, 3: motorcycle, 5: bus, 7: truck, 13: bench, 16: dog
TARGET_INDICES = [0, 1, 2, 3, 5, 7, 13, 16]
new_names = {i: orig_names[idx] for i, idx in enumerate(TARGET_INDICES)}
new_nc = len(TARGET_INDICES)

print(f"[INFO] Ursprüngliche Klassen: {len(orig_names)} -> Beschnitten auf: {new_nc}")
for i, name in new_names.items():
    print(f"  Index {i}: {name}")

# 3. Graph-Surgery am Detect-Kopf durchführen
detect_head = model.model.model[-1]
old_nc = detect_head.nc

for branch in detect_head.cv3:
    for m in branch.modules():
        if isinstance(m, nn.Conv2d) and m.out_channels == old_nc:
            m.weight = nn.Parameter(m.weight.data[TARGET_INDICES].clone())
            if m.bias is not None:
                m.bias = nn.Parameter(m.bias.data[TARGET_INDICES].clone())
            m.out_channels = new_nc

# Metadaten im PyTorch-Modell anpassen
detect_head.nc = new_nc
detect_head.no = new_nc + detect_head.reg_max * 4
model.model.nc = new_nc
model.model.names = new_names

# 4. Direkt als schlankes TFLite (320x320 FP32 für GPU-Delegate) exportieren
export_path = model.export(
    format="tflite",
    imgsz=320,
    simplify=True,
    nms=False
)

# 5. Lokalisieren und herunterladen
tflite_candidates = glob.glob("**/*yolo11n*.tflite", recursive=True) + glob.glob("**/*.tflite", recursive=True)

if tflite_candidates:
    target_file = max(tflite_candidates, key=os.path.getmtime)
    print(f"[SUCCESS] Lade exportiertes Modell herunter: {target_file}")
    files.download(target_file)
else:
    print(f"[HINWEIS] Manuell herunterladen von: {export_path}")

[INFO] Ursprüngliche Klassen: 80 -> Beschnitten auf: 8
  Index 0: person
  Index 1: bicycle
  Index 2: car
  Index 3: motorcycle
  Index 4: bus
  Index 5: truck
  Index 6: bench
  Index 7: dog
WARNING ⚠️ format='tflite' is deprecated as of 8.4.83 and has been replaced by the unified Google LiteRT format. Exporting format='litert' instead. See https://docs.ultralytics.com/integrations/litert
Ultralytics 8.4.152 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ This model has no one-to-one head; using one-to-many outputs.
Export to tflite in the cloud with Ultralytics Platform: https://platform.ultralytics.com


RuntimeError: The size of tensor a (8) must match the size of tensor b (80) at non-singleton dimension 0

In [ ]:
import glob
from ultralytics import YOLO
from google.colab import files

# 1. Standard-Modell laden (reine Detektion)
model = YOLO("yolo11n.pt")

# 2. Direkt als TFLite FP32 exportieren
export_path = model.export(
    format="tflite",
    imgsz=320,
    simplify=True,
    nms=False
)

# 3. Herunterladen
tflite_candidates = glob.glob("**/*yolo11n*.tflite", recursive=True) + glob.glob("**/*.tflite", recursive=True)
files.download(max(tflite_candidates, key=os.path.getmtime))

WARNING ⚠️ format='tflite' is deprecated as of 8.4.83 and has been replaced by the unified Google LiteRT format. Exporting format='litert' instead. See https://docs.ultralytics.com/integrations/litert
Ultralytics 8.4.152 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ This model has no one-to-one head; using one-to-many outputs.
YOLO11n summary (fused): 100 layers, 2,616,248 parameters, 0 gradients, 1.6 GFLOPs

PyTorch: starting from 'yolo11n.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 84, 2100) (5.4 MB)

LiteRT: starting export with litert_torch 0.9.4...


(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

(00:01) [START] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions

`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

(00:00) [START] Write Model to yolo11n.tflite

(00:00) [ DONE] Write Model to yolo11n.tflite (+00:00)

LiteRT: export success ✅ 12.7s, saved as 'yolo11n.tflite' (10.2 MB)

Export complete (13.4s)
Results saved to /content/yolo11n.tflite
Predict:         yolo predict task=detect model=yolo11n.tflite imgsz=320 
Validate:        yolo val task=detect model=yolo11n.tflite imgsz=320 data=/usr/src/ultralytics/ultralytics/cfg/datasets/coco.yaml  
Visualize:       https://netron.app


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>